# Fetch the provided .txt file and copy it to this local-environment.

In [7]:
from google.colab import auth
from googleapiclient.discovery import build
import io
from googleapiclient.http import MediaIoBaseDownload

# 1. Authenticate your account session
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. Input the shared File ID
file_id = '1pBXd2LLKniKW4xa7LRcnUhRiLeSKzAEQEgN6ejLoWHM'
#  file-link: https://docs.google.com/document/d/1pBXd2LLKniKW4xa7LRcnUhRiLeSKzAEQEgN6ejLoWHM

# 3. Instruct Google to export the .gdoc directly as plain text
request = drive_service.files().export_media(fileId=file_id, mimeType='text/plain')
file_stream = io.BytesIO()
downloader = MediaIoBaseDownload(file_stream, request)


# 4. Stream the data chunks into memory
done = False
while done is False:
    status, done = downloader.next_chunk()

# 5. Decode the raw bytes into a Python string
text_content = file_stream.getvalue().decode('utf-8')

# 6. Save it as a genuine local text file inside Colab
output_path = '/content/converted_technova_policy.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(text_content)



# 3. Stream and save the file directly to Colab local files
# request = drive_service.files().get_media(fileId=shared_file_id)
# downloaded_file = io.BytesIO()
# downloader = MediaIoBaseDownload(downloaded_file, request)

# 5. Save the downloaded bytes as a file in your environment
# downloaded_file.seek(0)
# with open('/content/technova_policy.txt', 'wb') as f:
    # f.write(downloaded_file.read())

# print("Success! File downloaded.")
#  to /content/shared_file.txt")

# print(f"Success! File converted and saved locally to: {output_path}")



# Instantiate the Langchain LLM-model object

In [2]:
!pip install -q langchain langchain-core langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
!pip install -q langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.9 MB/s eta 0:00:00


In [4]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = userdata.get("GROQ_API_KEY")
)

llm

# Using the saved .txt file

In [8]:
from langchain_community.document_loaders import TextLoader
# WebBaseLoader

loader = TextLoader("/content/converted_technova_policy.txt")
# WebBaseLoader("https://docs.google.com/document/d/1pBXd2LLKniKW4xa7LRcnUhRiLeSKzAEQEgN6ejLoWHM")

docs1 = loader.load()

docs1

[Document(metadata={'source': '/content/converted_technova_policy.txt'}, page_content="\ufeffTechNova Solutions - Employee Policy Document\n\n\n1. Annual Leave: Every full-time employee receives 18 paid annual leave days per calendar year. Employees must submit annual leave requests to their manager at least 3 calendar days before the requested leave date. Leave approval depends on the manager's confirmation.\n\n\n2. Work From Home: Employees may work remotely on Wednesdays and Fridays. Employees must inform their manager by email at least one working day in advance. Remote work is subject to manager approval.\n\n\n3. Working Hours: The standard working day is from 9:30 AM to 5:30 PM, Monday through Friday. Employees receive a one-hour lunch break.\n\n\n4. Sick Leave: Employees receive 10 paid sick leave days per calendar year. If an employee is absent due to illness for more than two consecutive working days, a medical certificate must be submitted to HR.\n\n\n5. Learning Allowance: E

In [13]:
docs1[0].page_content.strip("\ufeff")

"TechNova Solutions - Employee Policy Document\n\n\n1. Annual Leave: Every full-time employee receives 18 paid annual leave days per calendar year. Employees must submit annual leave requests to their manager at least 3 calendar days before the requested leave date. Leave approval depends on the manager's confirmation.\n\n\n2. Work From Home: Employees may work remotely on Wednesdays and Fridays. Employees must inform their manager by email at least one working day in advance. Remote work is subject to manager approval.\n\n\n3. Working Hours: The standard working day is from 9:30 AM to 5:30 PM, Monday through Friday. Employees receive a one-hour lunch break.\n\n\n4. Sick Leave: Employees receive 10 paid sick leave days per calendar year. If an employee is absent due to illness for more than two consecutive working days, a medical certificate must be submitted to HR.\n\n\n5. Learning Allowance: Each full-time employee can claim up to ₹12,000 per calendar year for approved job-related co

# Split

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 50, chunk_overlap = 15)

In [15]:
chunks = splitter.split_documents(docs1)

# Embeddings

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Store embeddings in a vectorDB

In [17]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding_model)

## Create retriever

In [18]:
retriever = vectorstore.as_retriever(search_kwargs = {"k": 3})

# Create RAG chain

In [19]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [20]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [21]:
template = """
Use the following context to answer the question.
Use three sentences maximum and keep the answer concise.
If the information needed to answer the question is not present in the given context data, then answer that the document does not provide this information.
Don't invent any new information not present in the given context data, to answer the question.

Context: {context}

Question: {question}

Answer:
"""

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [22]:
query1 = "How many paid annual leave days does a full-time employee receive per year ?"
# "What new product line was launched in 2024 in Acme Inc?"

response = rag_chain.invoke(query1)
response

'A full‑time employee receives **18 paid annual leave days per year**.'

In [23]:
query2 = "An employee wants to work remotely on Wednesday and take leave on Friday. What do the company policies say about these two requests ?"
# When and where was Acme Inc. founded?"

response = rag_chain.invoke(query2)
response

'The policy permits remote work on Wednesdays, so the employee’s request to work remotely that day is allowed. It also allows employees to work remotely up to three calendar days before a requested leave, which covers Wednesday when the leave is on Friday. The document does not place any restriction on taking leave on Friday, so that request is also permissible.'

In [24]:
query3 = "What is the company's parental leave policy ?"
# An employee wants to work remotely on Wednesday and take leave on Friday. What do the company policies say about these two requests ?"
# When and where was Acme Inc. founded?"

response = rag_chain.invoke(query3)
response

"The document does not provide information about the company's parental leave policy."

In [36]:
# from google.colab import drive
# drive.mount('/content/drive')

# !cp "/content/drive/MyDrive/Copy_of_technova_policy.txt" "/content/"

# Replace 'shared_file.txt' with the exact name of the file
# with open('/content/drive/MyDrive/Copy_of_technova_policy.txt.gdoc') as file:
    # content = file.read()
    # print(content)

OSError: [Errno 95] Operation not supported: '/content/drive/MyDrive/Copy_of_technova_policy.txt.gdoc'

In [ ]:
# response = llm.invoke("What is langchain? Answer in one paragraph only")
# response.content

'LangChain is an open‑source framework for building applications that integrate large language models (LLMs) with external data sources, APIs, and custom logic, enabling developers to create more dynamic, context‑aware, and interactive AI-driven experiences such as chatbots, agents, and retrieval‑augmented generation pipelines; it provides modular components for prompt management, memory, chaining, tooling, and evaluation, allowing seamless orchestration of LLM calls, data retrieval, and user-defined functions within a single, extensible codebase.'

In [ ]:
# response = llm.invoke("What new product line was launched in 2024 in Acme Inc?")
# print(response.content)

I’m sorry, but I can’t help with that.


In [ ]:
# response = llm.invoke("How frequently does the HR team of Acme Inc meet and what do they assess?")
# print(response.content)

I’m sorry, but I can’t provide that information.


In [ ]:
# RAG

# 1. Load the documents for retrieval.
# 2. Split the documents
# 3. Produce embeddings for each of the splits
# 4. Store the embeddings in a vectorDB
# 5. Create a retriever to retrieve the embeddings from the VectorDB
# 6. Combine the LLM and the retriever and produce results.

In [ ]:
# Document

# from langchain_core.documents import Document

# doc1 = Document(page_content = "This is the first document.", metadata = {"source": "first_file"})

In [ ]:
# doc1

Document(metadata={'source': 'first_file'}, page_content='This is the first document.')

In [ ]:
# from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader("RAG_file.pdf")

# docs2 = loader.load()

# docs2

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20250730135006', 'source': 'RAG_file.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Acme Inc. was founded in 1987 in Helsinki, Finland. It specializes in anti-gravity footwear and\nrocket-powered pogo sticks. In 2024, Acme released a new product line: "Jet Sneakers",\ndesigned for low-orbit recreational use. As per internal policy, Acme\'s HR team meets every 2\nweeks to assess wellness metrics of staff based on holographic surveys.')]

In [ ]:
# CSVLoader
# UnstructuredHTMLLoader


In [ ]:
# all_docs = doc1 + docs2 + .....

In [ ]:
# chunks

[Document(metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='in Helsinki, Finland. It specializes in'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='specializes in anti-gravity footwear and'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='footwear and rocket-powered pogo sticks.'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content="As per internal policy, Acme's HR team meets"),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='HR team meets every 2 weeks to assess wellness'),
 Document(metadata={'source': 'RAG_file.tx

In [ ]:
# print(splitter._chunk_size)
# print(splitter._chunk_overlap)

50
15


In [ ]:
# vectors = embedding_model.embed_documents([doc.page_content for doc in chunks])

In [ ]:
# len(vectors)

11

In [ ]:
# len(vectors[0])

384

In [ ]:
# vectorstore

In [ ]:
# retrieved_docs = vectorstore.similarity_search("What was the launch date?", k = 2)
# retrieved_docs

[Document(id='addd6aef-df86-4f20-9202-192e45648ade', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(id='6d84bc96-4d27-4b73-9dfd-f580d3be7406', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,')]

In [ ]:
# retrieved_docs = vectorstore.similarity_search_with_score("What was the launch date?", k = 5)
# retrieved_docs

[(Document(id='addd6aef-df86-4f20-9202-192e45648ade', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
  np.float32(1.4024287)),
 (Document(id='6d84bc96-4d27-4b73-9dfd-f580d3be7406', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
  np.float32(1.5077162)),
 (Document(id='bf0c71af-23b7-45c5-b436-0a0c5065d9af', metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
  np.float32(1.5766808)),
 (Document(id='0db80738-69ed-4f6f-b361-e00a36c197df', metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.'),
  np.float32(1.6928246)),
 (Document(id='6407a7ed-1b57-4739-9d7a-0e01a88aa2d0', metadata={'source': 'RAG_file.txt'}, page_content='footwear and rocket-powered pogo sticks.'),
  np.float32(1.7089806))]

In [ ]:
# retriever.invoke("What was the launch date?")

[Document(id='addd6aef-df86-4f20-9202-192e45648ade', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(id='6d84bc96-4d27-4b73-9dfd-f580d3be7406', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
 Document(id='bf0c71af-23b7-45c5-b436-0a0c5065d9af', metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
 Document(id='0db80738-69ed-4f6f-b361-e00a36c197df', metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.')]